# Frequency & Severity Modeling

**Goal:** fit a real frequency model (probability of a PTD or TTD claim, given
position and age) and a real severity model (games missed, given a claim
occurs), using GLMs -- the standard actuarial approach to loss modeling.

This picks up where the SQL/Excel rating tables left off: those showed
*empirical* frequency and severity by segment. This notebook fits an actual
statistical model, which lets us predict frequency/severity for any
position + age combination (not just the segments we happened to bucket),
and gives us statistically testable coefficients (are these differences
real, or noise?).

Data comes from the clean schema (`fact_player_seasons`, `fact_claims`,
`dim_players`) built in `scripts/06_build_clean_schema.py`.

In [1]:
import sqlite3
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

DB_PATH = "../data/nfl_injury_project.db"
conn = sqlite3.connect(DB_PATH)

## 1. Build the modeling dataset

One row per player-season, joined to birth date (to compute age) and to any
claim that occurred that season. `pending_immature` seasons (2023-2024, not
yet old enough to have a final PTD/TTD outcome) are excluded from model
fitting -- fitting on an outcome that hasn't been observed yet would bias
the model.

In [2]:
query = '''
SELECT
    ps.player_id, ps.season, ps.position_group, ps.age_band,
    p.birth_date,
    c.claim_classification, c.games_missed
FROM fact_player_seasons ps
JOIN dim_players p ON p.player_id = ps.player_id
LEFT JOIN fact_claims c ON c.player_id = ps.player_id AND c.season = ps.season
WHERE ps.age_band != 'Unknown'
'''
df = pd.read_sql(query, conn)

df["birth_date"] = pd.to_datetime(df["birth_date"], errors="coerce")
season_start = pd.to_datetime(df["season"].astype(str) + "-09-01")
df["age"] = (season_start - df["birth_date"]).dt.days / 365.25

df = df[df["claim_classification"] != "pending_immature"].copy()
df["is_ptd"] = (df["claim_classification"] == "PTD").astype(int)
df["is_ttd"] = (df["claim_classification"] == "TTD").astype(int)
df = df.dropna(subset=["age"])

print(f"Modeling dataset: {len(df):,} player-seasons")
df[["position_group","age","is_ptd","is_ttd"]].groupby("position_group").agg(
    n=("age","size"), avg_age=("age","mean"), ptd_rate=("is_ptd","mean"), ttd_rate=("is_ttd","mean")
).round(3)

Modeling dataset: 9,937 player-seasons


,n,avg_age,ptd_rate,ttd_rate
position_group,,,,
OL,5625,26.739,0.035,0.033
QB,1520,28.101,0.051,0.022
RB,2792,25.807,0.069,0.052


## 2. Frequency model: PTD

Logistic regression (GLM, binomial family) predicting the probability of a
PTD claim from position and age. OL is the reference category, since it's
the largest, most stable segment (same base-class logic used in the Excel
rating table).

In [3]:
model_ptd = smf.glm(
    'is_ptd ~ C(position_group, Treatment(reference="OL")) + age',
    data=df, family=sm.families.Binomial()
).fit()
print(model_ptd.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                 is_ptd   No. Observations:                 9937
Model:                            GLM   Df Residuals:                     9933
Model Family:                Binomial   Df Model:                            3
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -1850.7
Date:                Wed, 05 Aug 2026   Deviance:                       3701.5
Time:                        21:15:30   Pearson chi2:                 9.92e+03
No. Iterations:                     6   Pseudo R-squ. (CS):           0.008433
Covariance Type:            nonrobust                                         
                                                         coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------

**Reading the output:** the coefficients are in log-odds. Converting to
odds ratios makes them easier to interpret directly.

In [4]:
odds_ratios = np.exp(model_ptd.params)
conf_int = np.exp(model_ptd.conf_int())
summary_table = pd.DataFrame({
    "odds_ratio": odds_ratios,
    "ci_low": conf_int[0],
    "ci_high": conf_int[1],
    "p_value": model_ptd.pvalues,
})
summary_table.round(3)

,odds_ratio,ci_low,ci_high,p_value
Intercept,0.004,0.002,0.008,0.000
"C(position_group, Treatment(reference=""OL""))[T.QB]",1.281,0.973,1.687,0.078
"C(position_group, Treatment(reference=""OL""))[T.RB]",2.231,1.816,2.741,0.000
age,1.087,1.059,1.116,0.000


RB carries a significantly higher PTD odds ratio than the OL base class,
and each additional year of age meaningfully increases PTD odds -- both
consistent with the SQL/Excel findings, but now with actual confidence
intervals and p-values behind them rather than just point estimates.

## 3. Frequency model: TTD

Same structure, predicting TTD instead.

In [5]:
model_ttd = smf.glm(
    'is_ttd ~ C(position_group, Treatment(reference="OL")) + age',
    data=df, family=sm.families.Binomial()
).fit()
print(model_ttd.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                 is_ttd   No. Observations:                 9937
Model:                            GLM   Df Residuals:                     9933
Model Family:                Binomial   Df Model:                            3
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -1545.4
Date:                Wed, 05 Aug 2026   Deviance:                       3090.8
Time:                        21:15:30   Pearson chi2:                 9.97e+03
No. Iterations:                     7   Pseudo R-squ. (CS):           0.003129
Covariance Type:            nonrobust                                         
                                                         coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------

Interesting contrast with the PTD model: age's effect on TTD is *negative*
(not strongly significant here, p~0.08, but directionally consistent with
the SQL age-band finding) -- older players who suffer a significant injury
are relatively less likely to see it resolve as a "return" (TTD) and more
likely to see it end in PTD. This matches the loss development triangle
finding from the SQL layer.

## 4. Severity model: games missed, given a claim occurs

Games missed is a positive, continuous, right-skewed quantity -- a natural
fit for a Gamma GLM with a log link (a standard actuarial severity-model
choice), rather than ordinary least squares.

In [6]:
claims = df[df["claim_classification"].isin(["PTD","TTD"])].copy()
claims = claims[claims["games_missed"] > 0]

model_severity = smf.glm(
    'games_missed ~ C(position_group, Treatment(reference="OL")) + age + '
    'C(claim_classification, Treatment(reference="TTD"))',
    data=claims, family=sm.families.Gamma(sm.families.links.Log())
).fit()
print(model_severity.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:           games_missed   No. Observations:                  834
Model:                            GLM   Df Residuals:                      829
Model Family:                   Gamma   Df Model:                            4
Link Function:                    Log   Scale:                        0.078488
Method:                          IRLS   Log-Likelihood:                -2038.4
Date:                Wed, 05 Aug 2026   Deviance:                       66.759
Time:                        21:15:30   Pearson chi2:                     65.1
No. Iterations:                     9   Pseudo R-squ. (CS):             0.1072
Covariance Type:            nonrobust                                         
                                                                 coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------

**Key finding:** position and age are *not* strong drivers of severity once
a claim has already occurred -- but claim type is: PTD claims are
associated with meaningfully more games missed than TTD claims (as
expected, since PTD requires never returning). This is a genuinely useful
modeling result: frequency depends heavily on position/age, but severity
(once hurt) is driven more by *how bad* the injury is (PTD vs. TTD) than by
who the player is. That's a defensible, real actuarial insight for the
pricing memo: **segment heavily for frequency, less so for severity.**

## 5. Save fitted models for use in the pricing simulation notebook

In [7]:
import pickle
with open("fitted_models.pkl", "wb") as f:
    pickle.dump({"ptd": model_ptd, "ttd": model_ttd, "severity": model_severity}, f)
print("Saved fitted_models.pkl")
conn.close()

Saved fitted_models.pkl
